# 02. Data Warehouse – LYCOS-IDS2017

**Grain:** một dòng fact = một luồng mạng đã làm sạch. Kho dữ liệu hình sao lưu bằng SQLite (`artifacts/ids_warehouse.sqlite`), gồm fact luồng và các chiều thời gian, giao thức, cổng đích, nhãn. Cube tổng hợp theo ngày/giờ × giao thức × cổng × nhãn. Các câu hỏi nghiệp vụ được nêu trước mỗi truy vấn SQL và biểu đồ. Chạy notebook 01 trước.

Đây là dữ liệu thí nghiệm gắn nhãn, nên các chỉ số là phân tích lịch sử của bộ LYCOS-IDS2017, không phải tỷ lệ tấn công của một mạng đang vận hành. Cổng đích là proxy cho dịch vụ; một số luồng có chiều giao tiếp đảo nên cần diễn giải thận trọng.

## Import và đọc dữ liệu

In [ ]:
from pathlib import Path
import sqlite3

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
OUT_DIR = Path('artifacts')
DATA_FILE = OUT_DIR / 'flows_clean.parquet'
assert DATA_FILE.exists(), 'Hãy chạy 01_eda_preprocess.ipynb trước'
cols = ['timestamp', 'date', 'hour', 'ip_prot', 'src_port', 'dst_port',
        'label', 'is_attack', 'bytes_per_s', 'fwd_pkt_cnt', 'bwd_pkt_cnt']
df = pd.read_parquet(DATA_FILE, columns=cols)
print('Số luồng đưa vào warehouse:', len(df))
df.head()

## Thiết kế chiều và fact

`time_key` ở mức **giờ** (`YYYYMMDDHH`), `protocol_key` là mã IP protocol, `service_key` là cổng đích, `label_key` là nhãn chi tiết. Fact giữ `is_attack` và các số đo để tổng hợp. Bốn khóa chiều đều được lấy từ dữ liệu đã làm sạch; không có bảng chiều giả lập. Chỉ số `avg_bytes_per_s` là **trung bình** tốc độ trên luồng, không cộng tốc độ giữa các luồng.

In [ ]:
# date và hour đã tạo ở notebook 01; phép toán số nhanh hơn strftime trên 1,8 triệu dòng
df['time_key'] = df['date'].str.replace('-', '', regex=False).astype('int64') * 100 + df['hour'].astype('int64')
df['protocol_key'] = df['ip_prot'].astype('Int64')
df['service_key'] = df['dst_port'].astype('Int64')
df['label_key'] = df['label'].astype('string')

protocol_names = {1: 'ICMP', 2: 'IGMP', 6: 'TCP', 17: 'UDP'}
service_names = {20: 'FTP-data', 21: 'FTP', 22: 'SSH', 23: 'Telnet', 25: 'SMTP',
                 53: 'DNS', 80: 'HTTP', 110: 'POP3', 143: 'IMAP', 443: 'HTTPS',
                 445: 'SMB', 3389: 'RDP', 8080: 'HTTP-alt'}
dim_time = df[['time_key', 'date', 'hour']].drop_duplicates().sort_values('time_key')
dim_protocol = df[['protocol_key']].dropna().drop_duplicates().sort_values('protocol_key')
dim_protocol['protocol_name'] = dim_protocol['protocol_key'].map(protocol_names).fillna('Other')
dim_service = df[['service_key']].dropna().drop_duplicates().sort_values('service_key')
dim_service['service_name'] = dim_service['service_key'].map(service_names).fillna('Other')
dim_label = df[['label_key']].drop_duplicates().sort_values('label_key')
dim_label['class_group'] = dim_label['label_key'].eq('benign').map({True: 'Benign', False: 'Attack'})

fact_flow = df[['time_key', 'protocol_key', 'service_key', 'label_key',
                'is_attack', 'bytes_per_s', 'fwd_pkt_cnt', 'bwd_pkt_cnt']].copy()
display(pd.DataFrame({'Bảng': ['dim_time', 'dim_protocol', 'dim_service', 'dim_label', 'fact_flow'],
                      'Số dòng': [len(dim_time), len(dim_protocol), len(dim_service), len(dim_label), len(fact_flow)]}))

In [ ]:
# Tạo warehouse vật lý và cube materialized; chạy lại sẽ dựng mới để tránh nhân đôi fact
db_file = OUT_DIR / 'ids_warehouse.sqlite'
if db_file.exists():
    db_file.unlink()
con = sqlite3.connect(db_file)
for name, table in [('dim_time', dim_time), ('dim_protocol', dim_protocol),
                    ('dim_service', dim_service), ('dim_label', dim_label)]:
    table.to_sql(name, con, index=False, if_exists='replace')
fact_flow.to_sql('fact_flow', con, index=False, if_exists='replace', chunksize=50_000)
con.executescript('''
CREATE INDEX idx_fact_time ON fact_flow(time_key);
CREATE INDEX idx_fact_protocol ON fact_flow(protocol_key);
CREATE INDEX idx_fact_service ON fact_flow(service_key);
CREATE INDEX idx_fact_label ON fact_flow(label_key);
CREATE TABLE cube_flow AS
SELECT time_key, protocol_key, service_key, label_key,
       COUNT(*) AS flow_count,
       SUM(is_attack) AS attack_count,
       AVG(bytes_per_s) AS avg_bytes_per_s,
       SUM(fwd_pkt_cnt + bwd_pkt_cnt) AS packet_count
FROM fact_flow
GROUP BY time_key, protocol_key, service_key, label_key;
CREATE INDEX idx_cube_time ON cube_flow(time_key);
CREATE INDEX idx_cube_protocol ON cube_flow(protocol_key);
CREATE INDEX idx_cube_service ON cube_flow(service_key);
''')
con.commit()
print('Đã tạo:', db_file)
print('Số dòng cube:', pd.read_sql_query('SELECT COUNT(*) AS n FROM cube_flow', con).iloc[0, 0])

## Yêu cầu nghiệp vụ 1 – Giờ nào có tỷ lệ luồng tấn công cao?

Truy vấn cube theo **ngày × giờ**, tính `attack_count / flow_count`. Đây là tỷ lệ trong tập dữ liệu, không phải mức độ rủi ro ngoài thực tế.

In [ ]:
q1 = pd.read_sql_query('''
SELECT t.date, t.hour, SUM(c.flow_count) AS flows,
       SUM(c.attack_count) AS attacks,
       100.0 * SUM(c.attack_count) / SUM(c.flow_count) AS attack_pct
FROM cube_flow c JOIN dim_time t USING (time_key)
GROUP BY t.date, t.hour ORDER BY t.date, t.hour
''', con)
display(q1.head(10))
plt.figure(figsize=(14, 5))
sns.lineplot(data=q1, x='hour', y='attack_pct', hue='date', marker='o')
plt.title('Tỷ lệ luồng tấn công theo ngày và giờ')
plt.ylabel('Tỷ lệ tấn công (%)')
plt.show()

## Yêu cầu nghiệp vụ 2 – Giao thức nào có tỷ lệ tấn công cao trong mẫu?

In [ ]:
q2 = pd.read_sql_query('''
SELECT p.protocol_name, c.protocol_key, SUM(c.flow_count) AS flows,
       SUM(c.attack_count) AS attacks,
       100.0 * SUM(c.attack_count) / SUM(c.flow_count) AS attack_pct
FROM cube_flow c JOIN dim_protocol p USING (protocol_key)
GROUP BY c.protocol_key, p.protocol_name ORDER BY flows DESC
''', con)
display(q2)
plt.figure(figsize=(9, 4))
sns.barplot(data=q2, x='protocol_name', y='attack_pct', color='steelblue')
plt.title('Tỷ lệ tấn công theo giao thức IP')
plt.ylabel('Tỷ lệ tấn công (%)')
plt.show()

## Yêu cầu nghiệp vụ 3 – Cổng đích nào cần xem xét ưu tiên?

Chỉ xét cổng có ít nhất 100 luồng để tránh kết luận từ mẫu quá nhỏ. Trình bày 15 cổng có số luồng tấn công nhiều nhất, kèm mẫu số và tỷ lệ.

In [ ]:
q3 = pd.read_sql_query('''
SELECT s.service_key AS dst_port, s.service_name,
       SUM(c.flow_count) AS flows, SUM(c.attack_count) AS attacks,
       100.0 * SUM(c.attack_count) / SUM(c.flow_count) AS attack_pct
FROM cube_flow c JOIN dim_service s USING (service_key)
GROUP BY s.service_key, s.service_name
HAVING SUM(c.flow_count) >= 100
ORDER BY attacks DESC LIMIT 15
''', con)
display(q3)
plt.figure(figsize=(10, 5))
sns.barplot(data=q3, x='attacks', y=q3['dst_port'].astype(str) + ' (' + q3['service_name'] + ')', color='darkorange')
plt.title('Top 15 cổng đích theo số luồng tấn công')
plt.xlabel('Số luồng tấn công')
plt.ylabel('Cổng đích')
plt.show()

## Yêu cầu nghiệp vụ 4 – Loại tấn công nào xuất hiện theo từng ngày?

In [ ]:
q4 = pd.read_sql_query('''
SELECT t.date, l.label_key AS attack_type,
       SUM(c.flow_count) AS flows
FROM cube_flow c
JOIN dim_time t USING (time_key)
JOIN dim_label l USING (label_key)
WHERE l.class_group = 'Attack'
GROUP BY t.date, l.label_key ORDER BY t.date, flows DESC
''', con)
display(q4)
heat = q4.pivot(index='attack_type', columns='date', values='flows').fillna(0)
plt.figure(figsize=(11, 6))
sns.heatmap(heat, cmap='Blues', annot=True, fmt='.0f')
plt.title('Số luồng theo loại tấn công và ngày')
plt.show()

## Đối soát và lưu kết quả

Cube phải giữ đúng tổng số luồng và tấn công từ fact. Các truy vấn lưu thành CSV cho báo cáo. Những khác biệt lớn theo ngày có thể do **lịch kịch bản CIC-IDS2017**, không thể suy ra xu hướng vận hành thực tế.

**Kết quả chạy trên dữ liệu hiện có:** cube có 7.885 ô tổng hợp. Cả 441.823 luồng tấn công gắn nhãn đều thuộc TCP trong bộ này; UDP có 887.154 luồng nhưng không có nhãn tấn công. Đây là đặc điểm của mẫu thử, không có nghĩa UDP an toàn trong thực tế.

In [ ]:
check = pd.read_sql_query('''
SELECT (SELECT COUNT(*) FROM fact_flow) AS fact_flows,
       (SELECT SUM(flow_count) FROM cube_flow) AS cube_flows,
       (SELECT SUM(is_attack) FROM fact_flow) AS fact_attacks,
       (SELECT SUM(attack_count) FROM cube_flow) AS cube_attacks
''', con)
display(check)
assert check['fact_flows'].iloc[0] == check['cube_flows'].iloc[0] == len(df)
assert check['fact_attacks'].iloc[0] == check['cube_attacks'].iloc[0]
for name, result in [('q1_hourly', q1), ('q2_protocol', q2),
                     ('q3_ports', q3), ('q4_attack_type', q4)]:
    result.to_csv(OUT_DIR / f'{name}.csv', index=False)
con.close()